# Fisheries Production and Market Analysis in India

EDAV Project

This project compares fisheries production across **29 Indian states**, classifies each state as **Coastal** or **Inland**, explores species-level patterns, and then looks at **monthly trends (time series)** and the **market side** (prices and revenue).

- **State dataset:** one row per state. Species and total values are average monthly production across the available 2022–2023 records; they are not annual totals. 9 inland states are flagged `Is_Estimated = True` because their values were estimated, not taken from official records.
- **Monthly dataset:** `fisheries_monthly_simulated.csv` has 24 months (2022–2023) per state with production and prices. It is **simulated** for this project: monthly values average out to the state dataset, with seasonal patterns (such as the monsoon fishing ban) and prices added. It is not official data, so the time series and market results are illustrative.

## 1. Import Libraries

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 6)

## 2. Load Dataset

In [ ]:
# In Google Colab, upload FISHY.csv when prompted.
try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
    raw = pd.read_csv(io.BytesIO(uploaded[csv_name]))
except ImportError:
    raw = pd.read_csv("FISHY.csv")

raw.head()

## 3. View Dataset

In [ ]:
print("Rows and columns:", raw.shape)
print("Unique states:", raw["State"].nunique())
display(raw.head(10))
display(raw.dtypes.to_frame("Data type"))

## 4. Data Cleaning

In [ ]:
data = raw.copy()
data.columns = data.columns.str.strip()
data["State"] = data["State"].astype("string").str.strip()
data["Type"] = data["Type"].astype("string").str.strip().str.title()
data["Is_Estimated"] = data["Is_Estimated"].astype(bool)

numeric_cols = data.select_dtypes(include="number").columns.tolist()
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data = data.drop_duplicates(subset="State").sort_values("State").reset_index(drop=True)
print("Shape after cleaning:", data.shape)
print("States represented once:", data["State"].is_unique)
print("Estimated states:", data.loc[data["Is_Estimated"], "State"].tolist())
display(data.isna().sum().to_frame("Missing values"))

### Missing Values

In [ ]:
# Report missing values rather than silently replacing them with invented observations.
missing = data.isna().sum().sort_values(ascending=False)
display(missing.to_frame("Missing values"))

## 5. Prepare Data

In [ ]:
species_cols = [c for c in data.columns if c not in ["State", "Type", "Total_Tonnes", "Is_Estimated"]]

# Group species: Hilsa is migratory but is caught mainly in marine/estuarine waters, so it is grouped with Marine.
freshwater_cols = ["Rohu", "Catla", "Tilapia", "Mrigal", "Silver_Carp", "Pangas"]
marine_cols = ["Pomfret", "Sardine", "Mackerel", "Shrimp", "Tuna", "Hilsa"]
data["Freshwater_Tonnes"] = data[freshwater_cols].sum(axis=1)
data["Marine_Tonnes"] = data[marine_cols].sum(axis=1)

print("State categories:")
display(data["Type"].value_counts().to_frame("Number of states"))
print("Species columns:", species_cols)

## 6. NumPy Operations

In [ ]:
production_1d = data["Total_Tonnes"].dropna().to_numpy()
production_2d = data[species_cols].to_numpy(dtype=float)

print("1D array shape:", production_1d.shape)
print("2D array shape (states x species):", production_2d.shape)
print("Mean of state-level average production:", np.nanmean(production_1d).round(2))
print("Maximum species value:", np.nanmax(production_2d).round(2))

## 7. Advanced NumPy

In [ ]:
print("Transposed shape:", production_2d.T.shape)
flat_values = production_2d.ravel()
print("Flattened array length:", len(flat_values))
print("First 10 flattened values:", flat_values[:10])

## 8. Pandas Operations

In [ ]:
print("Series type:", type(data["Total_Tonnes"]))
print("DataFrame type:", type(data))
display(data[["State", "Type", "Total_Tonnes"]].sort_values("Total_Tonnes", ascending=False))

## 9. Data Transformation

In [ ]:
long_data = data.melt(
    id_vars=["State", "Type"],
    value_vars=species_cols,
    var_name="Species",
    value_name="Average_Monthly_Production"
)
display(long_data.head(10))

## 10. GroupBy and Aggregation

In [ ]:
type_summary = data.groupby("Type", as_index=False).agg(
    States=("State", "nunique"),
    Mean_Production=("Total_Tonnes", "mean"),
    Median_Production=("Total_Tonnes", "median")
)
species_summary = long_data.groupby("Species", as_index=False).agg(
    Average_Production=("Average_Monthly_Production", "mean"),
    Highest_State_Value=("Average_Monthly_Production", "max")
).sort_values("Average_Production", ascending=False)
display(type_summary.round(2))
display(species_summary.round(2))

## 11. Indexing, Merge and Pivot

In [ ]:
indexed_data = data.set_index(["Type", "State"]).sort_index()
display(indexed_data.head(8))

state_species_pivot = long_data.pivot_table(
    index="State", columns="Species", values="Average_Monthly_Production", aggfunc="mean"
)
display(state_species_pivot.head())

## 12. Coastal vs Inland Comparison

In [ ]:
group_summary = data.groupby("Type")[["Total_Tonnes", "Freshwater_Tonnes", "Marine_Tonnes"]].agg(["mean", "median"]).round(1)
display(group_summary)

# Same comparison without the estimated states
display(data[~data["Is_Estimated"]].groupby("Type")["Total_Tonnes"].agg(["count", "mean", "median"]).round(1))

# Boxplot of freshwater vs marine production by state type
box_data = data.melt(id_vars=["State", "Type"], value_vars=["Freshwater_Tonnes", "Marine_Tonnes"],
                     var_name="Group", value_name="Tonnes")
box_data["Group"] = box_data["Group"].str.replace("_Tonnes", "")
plt.figure(figsize=(8, 5))
sns.boxplot(data=box_data, x="Group", y="Tonnes", hue="Type", palette={"Coastal": "#20B2AA", "Inland": "#FF8C69"})
sns.stripplot(data=box_data, x="Group", y="Tonnes", hue="Type", dodge=True, palette={"Coastal": "black", "Inland": "black"}, size=3, legend=False)
plt.title("Freshwater vs Marine Production: Coastal vs Inland States", fontsize=14, weight="bold")
plt.ylabel("Average monthly production (tonnes)")
plt.tight_layout()
plt.show()

## 13. Matplotlib Graphs

In [ ]:
# Horizontal bar chart coloured by state type; estimated states are hatched
from matplotlib.patches import Patch
state_totals = data.sort_values("Total_Tonnes", ascending=True)
type_colors = {"Coastal": "#20B2AA", "Inland": "#FF8C69"}
plt.figure(figsize=(12, 10))
bars = plt.barh(state_totals["State"], state_totals["Total_Tonnes"],
                color=state_totals["Type"].map(type_colors))
for bar, est in zip(bars, state_totals["Is_Estimated"]):
    if est:
        bar.set_hatch("//")
        bar.set_edgecolor("white")
plt.title("Average Monthly Fisheries Production by State", fontsize=15, weight="bold")
plt.xlabel("Average monthly production (tonnes)")
plt.ylabel("State")
plt.legend(handles=[Patch(color=type_colors["Coastal"], label="Coastal"),
                    Patch(color=type_colors["Inland"], label="Inland"),
                    Patch(facecolor="grey", hatch="//", edgecolor="white", label="Estimated values")],
           loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Coastal vs inland: median production (less affected by one large state than the mean)
type_med = data.groupby("Type", as_index=False)["Total_Tonnes"].median()
plt.figure(figsize=(8, 5))
bars = plt.bar(type_med["Type"], type_med["Total_Tonnes"], color=[type_colors[t] for t in type_med["Type"]])
plt.title("Median Monthly Production: Coastal vs Inland", fontsize=14, weight="bold")
plt.ylabel("Median monthly production (tonnes)")
plt.xlabel("State category")
plt.bar_label(bars, fmt="%.1f", padding=3)
plt.tight_layout()
plt.show()

## 14. Seaborn Graphs

In [ ]:
species_plot = species_summary.sort_values("Average_Production", ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(data=species_plot, x="Species", y="Average_Production", hue="Species", palette="Spectral", legend=False)
plt.title("Average Production by Species", fontsize=14, weight="bold")
plt.xlabel("Species")
plt.ylabel("Average monthly production")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 15. Heatmap

In [ ]:
heatmap_data = data.set_index("State")[species_cols]
plt.figure(figsize=(13, 10))
sns.heatmap(heatmap_data, cmap="YlGnBu", linewidths=0.25, annot=False)
plt.title("State-wise Average Monthly Production by Species", fontsize=14, weight="bold")
plt.xlabel("Species")
plt.ylabel("State")
plt.tight_layout()
plt.show()

## 16. Time Series and Market Analysis

Monthly production and price data (simulated, 2022–2023), merged with the state dataset to get each state's type.

In [ ]:
# In Google Colab, upload fisheries_monthly_simulated.csv when prompted.
try:
    from google.colab import files
    up = files.upload()
    monthly = pd.read_csv(io.BytesIO(up[next(iter(up))]))
except ImportError:
    monthly = pd.read_csv("fisheries_monthly_simulated.csv")

monthly["Month"] = pd.to_datetime(monthly["Month"])
monthly = monthly.merge(data[["State", "Type", "Is_Estimated"]], on="State", how="left")

# Revenue in Rs crore: tonnes x 1000 kg x Rs/kg, divided by 1e7
monthly["Revenue_Cr"] = (monthly["Freshwater_Tonnes"] * monthly["Freshwater_Price_Rs_kg"]
                         + monthly["Marine_Tonnes"].fillna(0) * monthly["Marine_Price_Rs_kg"].fillna(0)) * 1000 / 1e7
monthly["Total_Tonnes_M"] = monthly["Freshwater_Tonnes"] + monthly["Marine_Tonnes"]
print(monthly.shape)
display(monthly.head())

In [ ]:
# National monthly production: freshwater vs marine, with 3-month rolling average
national = monthly.groupby("Month")[["Freshwater_Tonnes", "Marine_Tonnes"]].sum()
plt.figure(figsize=(11, 5))
for col, color in [("Freshwater_Tonnes", "#FF8C69"), ("Marine_Tonnes", "#20B2AA")]:
    plt.plot(national.index, national[col], marker="o", alpha=0.4, color=color)
    plt.plot(national.index, national[col].rolling(3).mean(), linewidth=3, color=color, label=col.replace("_Tonnes", "") + " (3-month avg)")
plt.title("Monthly Production Across All States", fontsize=14, weight="bold")
plt.ylabel("Production (tonnes)")
plt.legend()
plt.tight_layout()
plt.show()

# Seasonality: average by calendar month
seasonal = monthly.groupby(monthly["Month"].dt.month)[["Freshwater_Tonnes", "Marine_Tonnes"]].sum() / 2
display(seasonal.round(0))

In [ ]:
# Price vs production: weighted average price per kg each month
agg = monthly.groupby("Month").apply(lambda g: pd.Series({
    "Production": g["Total_Tonnes_M"].sum(),
    "Avg_Price": (g["Revenue_Cr"].sum() * 1e7) / (g["Total_Tonnes_M"].sum() * 1000),
    "Revenue_Cr": g["Revenue_Cr"].sum()
}), include_groups=False)
corr = agg["Production"].corr(agg["Avg_Price"])
print("Correlation between monthly production and average price:", round(corr, 2))

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(agg.index, agg["Production"], width=20, color="#20B2AA", alpha=0.6, label="Production (tonnes)")
ax1.set_ylabel("Production (tonnes)")
ax2 = ax1.twinx()
ax2.plot(agg.index, agg["Avg_Price"], color="#D2691E", marker="o", linewidth=2.5, label="Average price (Rs/kg)")
ax2.set_ylabel("Average price (Rs/kg)")
ax2.grid(False)
plt.title("Monthly Production and Average Price", fontsize=14, weight="bold")
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.92))
plt.tight_layout()
plt.show()

In [ ]:
# Revenue: coastal vs inland, and by year
monthly["Year"] = monthly["Month"].dt.year
rev = monthly.groupby(["Year", "Type"])["Revenue_Cr"].sum().unstack()
display(rev.round(0))
rev.plot(kind="bar", color=["#20B2AA", "#FF8C69"], figsize=(8, 5), rot=0)
plt.title("Total Revenue by Year: Coastal vs Inland", fontsize=14, weight="bold")
plt.ylabel("Revenue (Rs crore)")
plt.tight_layout()
plt.show()

top_rev = monthly.groupby("State")["Revenue_Cr"].sum().sort_values(ascending=False).head(5)
display(top_rev.round(0).to_frame("Total revenue (Rs crore)"))

### Time Series and Market Findings

*These come from the simulated monthly data, so they show what the analysis reveals given the seasonal and price patterns built into it, not real market behaviour.*

1. **Marine production is strongly seasonal.** It falls to its lowest in June (about 2,300 tonnes nationally) during the monsoon fishing ban and peaks in October and November (about 8,400 tonnes).
2. **Freshwater production is steadier,** staying between about 19,200 and 23,100 tonnes a month and peaking in October.
3. **Prices tend to move against supply.** The correlation between monthly production and average price is weak and negative (about -0.32), so prices are somewhat higher in low-supply months.
4. **Coastal states earn more total revenue** (about Rs 3,900 crore in 2023 against Rs 3,100 crore for inland), even though the 20 inland states produce slightly more tonnage in total, because marine species sell at higher prices.
5. **Revenue grew about 6–7% from 2022 to 2023** for both groups, which mostly reflects the price increase assumed in the data.

## 17. Results and Insights

In [ ]:
top_state = data.loc[data["Total_Tonnes"].idxmax()]
top_species = species_summary.iloc[0]
print(f"Highest average monthly production: {top_state['State']} ({top_state['Total_Tonnes']:.1f} tonnes)")
print(f"State category: {top_state['Type']}")
print(f"Highest average species value across states: {top_species['Species']} ({top_species['Average_Production']:.1f})")
print("\nAverage monthly production by category:")
display(type_summary.round(2))

### Key Findings

1. **Kerala** has the highest average monthly production (1,545.2 tonnes) and **Haryana** the lowest (598.9 tonnes).
2. **Coastal states produce about twice as much as inland states.** The coastal median is 1,457.5 tonnes against 709.4 for inland, and the difference is statistically significant (Mann-Whitney U test, p < 0.001). Every coastal state (lowest: Goa, 1,362.2) is above every inland state (highest: Jharkhand, 837.7).
3. **The gap comes entirely from marine species.** Freshwater output is nearly the same in both groups (about 723 tonnes for coastal and 716 for inland on average), while coastal states add about 726 tonnes of marine species on top. Marine species make up about half of coastal production.
4. **Inland production rests on carps and Pangas.** Catla, Silver Carp and Pangas lead in inland states, and Tilapia is the lowest of the six freshwater species.
5. **The estimated states do not drive the result.** Inland states with estimated values have a median of 721.0 tonnes against 706.6 for those with reported values, so the coastal vs inland conclusion holds either way. Still, the 9 estimated states should be replaced with official figures (for example from the DAHD Handbook on Fisheries Statistics) before any formal use.

## 18. Limitations

- The state CSV has **29 rows: one row per state**, with average monthly values from the available 2022–2023 data, not annual totals.
- **9 inland states have estimated values** (flagged `Is_Estimated = True`: Arunachal Pradesh, Himachal Pradesh, Jammu and Kashmir, Manipur, Meghalaya, Mizoram, Nagaland, Sikkim, Uttarakhand). They should not be presented as official government statistics.
- **The monthly production and price data is simulated.** Time series, seasonality, price and revenue results show the method and are illustrative only. Replace `fisheries_monthly_simulated.csv` with real monthly and price data (for example from NFDB, DAHD or Agmarknet) before drawing real conclusions.
- Only 9 states are coastal, so the coastal vs inland comparison rests on a small, unbalanced sample.
- Prices are given per species group (freshwater and marine), not per species, and there is no trade data.